In [7]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

# ─────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────
IMG_SIZE    = 128
NUM_CLASSES = 10   # 0-Neutral,1-Red,2-Orange,3-Yellow,4-Green,
                   # 5-Cyan,6-Blue,7-Violet,8-Pink,9-Brown
NUM_SAMPLES = 3000

CLASS_NAMES = [
    'Neutral', 'Red', 'Orange', 'Yellow', 'Green',
    'Cyan',    'Blue','Violet', 'Pink',   'Brown'
]

# ─────────────────────────────────────────────
# HSV → RGB (pure Python, no OpenCV dependency)
# h in [0,179]  s,v in [0,255]  — OpenCV convention
# ─────────────────────────────────────────────
def hsv_to_rgb(h, s, v):
    h_f = float(h) / 179.0 * 360.0   # → [0, 360)
    s_f = float(s) / 255.0
    v_f = float(v) / 255.0

    if s_f == 0.0:
        c = int(v_f * 255)
        return np.array([c, c, c], dtype=np.uint8)

    hi  = int(h_f / 60) % 6
    f   = h_f / 60.0 - int(h_f / 60)
    p   = v_f * (1.0 - s_f)
    q   = v_f * (1.0 - f * s_f)
    t   = v_f * (1.0 - (1.0 - f) * s_f)

    if   hi == 0: r, g, b = v_f, t,   p
    elif hi == 1: r, g, b = q,   v_f, p
    elif hi == 2: r, g, b = p,   v_f, t
    elif hi == 3: r, g, b = p,   q,   v_f
    elif hi == 4: r, g, b = t,   p,   v_f
    else:         r, g, b = v_f, p,   q

    return np.array([int(r * 255), int(g * 255), int(b * 255)], dtype=np.uint8)

# ─────────────────────────────────────────────
# Per-pixel class assignment (HSV rules)
# ─────────────────────────────────────────────
def assign_class(h, s, v):
    """Assign a pixel to one of 10 color classes based on OpenCV HSV values."""
    if s < 40 or v < 40:
        return 0
    if 11 <= h <= 25 and s >= 80 and 40 <= v <= 120:   # Brown (before Orange)
        return 9
    if (h <= 10 or h >= 170) and s >= 80 and v >= 80:  # Red
        return 1
    if 11 <= h <= 25 and s >= 80 and v > 120:           # Orange
        return 2
    if 26 <= h <= 38 and s >= 60 and v >= 100:          # Yellow
        return 3
    if 39 <= h <= 80 and s >= 60 and v >= 60:           # Green
        return 4
    if 81 <= h <= 100 and s >= 60 and v >= 80:          # Cyan
        return 5
    if 101 <= h <= 130 and s >= 60 and v >= 60:         # Blue
        return 6
    if 131 <= h <= 155 and s >= 60 and v >= 60:         # Violet
        return 7
    if 156 <= h <= 169 and 30 <= s <= 120 and v >= 150: # Pink
        return 8
    return 0

# ─────────────────────────────────────────────
# Representative HSV centres for each non-neutral class
# (h_center, s, v, class_id)
# ─────────────────────────────────────────────
CLASS_SEEDS = [
    (5,   200, 200, 1),   # Red
    (18,  210, 210, 2),   # Orange
    (32,  200, 215, 3),   # Yellow
    (60,  200, 150, 4),   # Green
    (90,  200, 200, 5),   # Cyan
    (115, 200, 160, 6),   # Blue
    (143, 200, 160, 7),   # Violet
    (162,  80, 220, 8),   # Pink
    (18,  200,  80, 9),   # Brown
]

# ─────────────────────────────────────────────
# Sample generator
# ─────────────────────────────────────────────
def generate_sample():
    image = np.full((IMG_SIZE, IMG_SIZE, 3), 180, dtype=np.uint8)
    mask  = np.zeros((IMG_SIZE, IMG_SIZE),   dtype=np.uint8)

    # Explicitly cast to Python int to avoid numpy scalar issues
    num_rects = int(np.random.randint(4, 8))
    for _ in range(num_rects):
        idx   = int(np.random.randint(0, len(CLASS_SEEDS)))
        h_c, s_c, v_c, cls_id = CLASS_SEEDS[idx]

        h = int(np.clip(h_c + int(np.random.randint(-8, 9)),   0, 179))
        s = int(np.clip(s_c + int(np.random.randint(-20, 21)), 0, 255))
        v = int(np.clip(v_c + int(np.random.randint(-20, 21)), 0, 255))

        rgb = hsv_to_rgb(h, s, v)

        x1 = int(np.random.randint(0, IMG_SIZE - 20))
        y1 = int(np.random.randint(0, IMG_SIZE - 20))
        w  = int(np.random.randint(15, IMG_SIZE // 2))
        h_ = int(np.random.randint(15, IMG_SIZE // 2))
        x2 = min(x1 + w,  IMG_SIZE)
        y2 = min(y1 + h_, IMG_SIZE)

        image[y1:y2, x1:x2] = rgb
        mask[y1:y2,  x1:x2] = cls_id

    return image, mask

# ─────────────────────────────────────────────
# Dataset generation
# ─────────────────────────────────────────────
print("Generating dataset…")
images, masks = [], []
for _ in range(NUM_SAMPLES):
    img, msk = generate_sample()
    images.append(img)
    masks.append(msk)

X = np.array(images, dtype=np.float32) / 255.0
y = np.expand_dims(np.array(masks, dtype=np.int32), axis=-1)

split    = int(0.8 * NUM_SAMPLES)
X_train  = X[:split];  X_test  = X[split:]
y_train  = y[:split];  y_test  = y[split:]
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# ─────────────────────────────────────────────
# U-Net with BatchNormalization (32 / 64 / 128 filters)
# ─────────────────────────────────────────────
def build_unet(num_classes=NUM_CLASSES):
    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))

    # Encoder
    c1 = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Conv2D(32, 3, activation='relu', padding='same')(c1)
    c1 = layers.BatchNormalization()(c1)
    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(64, 3, activation='relu', padding='same')(p1)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Conv2D(64, 3, activation='relu', padding='same')(c2)
    c2 = layers.BatchNormalization()(c2)
    p2 = layers.MaxPooling2D()(c2)

    # Bottleneck
    b = layers.Conv2D(128, 3, activation='relu', padding='same')(p2)
    b = layers.BatchNormalization()(b)
    b = layers.Conv2D(128, 3, activation='relu', padding='same')(b)
    b = layers.BatchNormalization()(b)

    # Decoder
    u1 = layers.UpSampling2D()(b)
    u1 = layers.Concatenate()([u1, c2])
    c3 = layers.Conv2D(64, 3, activation='relu', padding='same')(u1)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Conv2D(64, 3, activation='relu', padding='same')(c3)
    c3 = layers.BatchNormalization()(c3)

    u2 = layers.UpSampling2D()(c3)
    u2 = layers.Concatenate()([u2, c1])
    c4 = layers.Conv2D(32, 3, activation='relu', padding='same')(u2)
    c4 = layers.BatchNormalization()(c4)
    c4 = layers.Conv2D(32, 3, activation='relu', padding='same')(c4)
    c4 = layers.BatchNormalization()(c4)

    outputs = layers.Conv2D(num_classes, 1, activation='softmax')(c4)
    return models.Model(inputs, outputs)

model = build_unet()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

# ─────────────────────────────────────────────
# Class weights (reduce Neutral dominance)
# ─────────────────────────────────────────────
class_weight = {0: 0.3}
for i in range(1, NUM_CLASSES):
    class_weight[i] = 2.0

# ─────────────────────────────────────────────
# Training
# ─────────────────────────────────────────────
print("Training model…")
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=16,
    validation_data=(X_test, y_test),
    class_weight=class_weight
)

# ─────────────────────────────────────────────
# Quick visualisation
# ─────────────────────────────────────────────
test_img, test_mask = generate_sample()
pred_raw  = model.predict(np.expand_dims(test_img / 255.0, 0))
pred_mask = np.argmax(pred_raw[0], axis=-1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1); plt.title("Input"); plt.imshow(test_img)
plt.subplot(1, 3, 2); plt.title("Ground Truth"); plt.imshow(test_mask, cmap='tab10', vmin=0, vmax=9)
plt.subplot(1, 3, 3); plt.title("Predicted"); plt.imshow(pred_mask,  cmap='tab10', vmin=0, vmax=9)
plt.show()


Generating dataset…
Train: (2400, 128, 128, 3), Test: (600, 128, 128, 3)


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_55 (Conv2D)  │ (None, 128, 128,  │        896 │ input_layer_5[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        128 │ conv2d_55[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_56 (Conv2D)  │ (None, 128, 128,  │      9,248 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        128 │ conv2d_56[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_10    │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_57 (Conv2D)  │ (None, 64, 64,    │     18,496 │ max_pooling2d_10… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_57[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_58 (Conv2D)  │ (None, 64, 64,    │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_58[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_11    │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_59 (Conv2D)  │ (None, 32, 32,    │     73,856 │ max_pooling2d_11… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        512 │ conv2d_59[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_60 (Conv2D)  │ (None, 32, 32,    │    147,584 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        512 │ conv2d_60[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_10    │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (UpSampling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_10      │ (None, 64, 64,    │          0 │ up_sampling2d_10

 Total params: 474,410 (1.81 MB)

 Trainable params: 473,130 (1.80 MB)

 Non-trainable params: 1,280 (5.00 KB)

Training model…


TypeError: only length-1 arrays can be converted to Python scalars

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TFLITE INT8 QUANTIZATION AND EXPORT
# Run this cell AFTER training is complete.
# ═══════════════════════════════════════════════════════════════
import os

# ── 1. Save Keras model ──────────────────────────────────────
model.save('color_segmenter.keras')
print("Keras model saved.")

# ── 2. Calibration dataset for INT8 quantization ─────────────
def representative_data_gen():
    for _ in range(200):
        img, _ = generate_sample()
        yield [np.expand_dims(img / 255.0, axis=0).astype(np.float32)]

# ── 3. Convert with weight-only INT8 quantization ─────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations            = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset   = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# Keep float32 I/O so the JS side doesn't need integer de-quantization
converter.inference_input_type  = tf.float32
converter.inference_output_type = tf.float32

tflite_model = converter.convert()

# ── 4. Save ──────────────────────────────────────────────────
output_path = 'color_model.tflite'
with open(output_path, 'wb') as f:
    f.write(tflite_model)
print(f"TFLite model saved: {len(tflite_model)/1024:.1f} KB  →  {output_path}")

# ── 5. Verify shapes ─────────────────────────────────────────
interpreter = tf.lite.Interpreter(model_path=output_path)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]
print(f"Input  : shape={inp['shape']}, dtype={inp['dtype']}")   # [1,128,128,3] float32
print(f"Output : shape={out['shape']}, dtype={out['dtype']}")   # [1,128,128,10] float32

# ── 6. Copy to React Native assets ───────────────────────────
assets_dir = os.path.join('..', 'assets')
if os.path.isdir(assets_dir):
    import shutil
    dest = os.path.join(assets_dir, 'color_model.tflite')
    shutil.copy(output_path, dest)
    print(f"Copied to {dest}")
else:
    print(f"Note: copy {output_path} manually to the assets/ folder in the React Native project.")
